# Safety and Validation Strategies for SQL

**ADD RISKS DIAGRAM**



## Admin Access Settings

**Add DIAGRAM**

## Python Client

- `sql`
- `raw_sql`

In [40]:
import sys
import os

tbl_name = "air_traffic"

current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from sql_ai_agent.data import get_ibis_connection

csv_path = project_root + "/data/air_traffic_gold.csv"
con = get_ibis_connection(
    backend="duckdb",
    duckdb_csv_path=csv_path,
)

In [41]:
output = None
try:
    output = con.sql("SELECT * FROM air_traffic").execute()
except Exception as e:
    print("Could not execute the query")
    print(e)

output

,Unnamed: 0,Year,Date,Operating Airline,Operating Airline IATA Code,Published Airline,Published Airline IATA Code,GEO Summary,GEO Region,Activity Type Code,Price Category Code,Terminal,Boarding Area,Passenger Count
0,0,1999,1999-07-01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Deplaned,Low Fare,Terminal 1,B,31432
1,1,1999,1999-07-01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Enplaned,Low Fare,Terminal 1,B,31353
2,2,1999,1999-07-01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Thru / Transit,Low Fare,Terminal 1,B,2518
3,3,1999,1999-07-01,Aeroflot Russian International Airlines,None,Aeroflot Russian International Airlines,None,International,Europe,Deplaned,Other,Terminal 2,D,1324
4,4,1999,1999-07-01,Aeroflot Russian International Airlines,None,Aeroflot Russian International Airlines,None,International,Europe,Enplaned,Other,Terminal 2,D,1198
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38541,38541,2025,2025-07-01,Virgin Atlantic,VS,Virgin Atlantic,VS,International,Europe,Enplaned,Other,International,A,13178
38542,38542,2025,2025-07-01,WestJet,WS,WestJet,WS,International,Canada,Deplaned,Other,International,A,14451
38543,38543,2025,2025-07-01,WestJet,WS,WestJet,WS,International,Canada,Enplaned,Other,International,A,12475
38544,38544,2025,2025-07-01,ZIPAIR Tokyo Inc,ZG,ZIPAIR Tokyo Inc,ZG,International,Asia,Deplaned,Other,International,A,8602


In [36]:
try:
  con.sql("DROP TABLE air_traffic").execute()
except Exception as e:
    print("Could not execute the query")
    print(e)


Could not execute the query
Parser Error: syntax error at or near "TABLE"

LINE 1: DESCRIBE DROP TABLE air_traffic
                      ^


In [43]:
try:
    con.raw_sql("DROP TABLE air_traffic")
except Exception as e:
    print("Could not execute the query")
    print(e)


In [44]:

try:
    con.sql("SELECT * FROM air_traffic").execute()
except Exception as e:
    print("Could not execute the query")
    print(e)


Could not execute the query
Catalog Error: Table with name air_traffic does not exist!
Did you mean "pg_constraint"?

LINE 1: DESCRIBE SELECT * FROM air_traffic
                               ^


## SQL Validation

In [39]:
from typing import Iterable, Type

from sqlglot import parse, exp
from sqlglot.expressions import Expression


def is_sql_allowed(
    sql: str,
    *,
    allowed_statements: Iterable[Type[Expression]],
    disallowed_statements: Iterable[Type[Expression]],
    dialect: str | None = None,
) -> bool:
    """
    Validate a SQL query using SQLGlot against allow / deny expression policies.

    Parameters
    ----------
    sql : str
        SQL query to validate
    allowed_statements : Iterable[Type[Expression]]
        Expression types allowed at the top level
    disallowed_statements : Iterable[Type[Expression]]
        Expression types that are forbidden anywhere in the AST
    dialect : str, optional
        SQL dialect (e.g. "duckdb", "postgres", "snowflake")

    Returns
    -------
    bool
        True if the SQL query complies with the policy, False otherwise
    """
    try:
        statements = parse(sql, dialect=dialect)
    except Exception:
        # Invalid SQL → reject
        return False

    allowed_statements = tuple(allowed_statements)
    disallowed_statements = tuple(disallowed_statements)

    for statement in statements:
        # 1. Enforce top-level allowlist
        if not isinstance(statement, allowed_statements):
            return False

        # 2. Reject forbidden expressions anywhere in the AST
        for node in statement.walk():
            if isinstance(node, disallowed_statements):
                return False

    return True


In [45]:
READ_ONLY_ALLOWED = (
    exp.Select,
    exp.With,
    exp.Except,
    exp.Show,
    exp.Describe,
)

READ_ONLY_DISALLOWED = (
    exp.Insert,
    exp.Update,
    exp.Delete,
    exp.Create,
    exp.Drop,
    exp.Alter,
    exp.TruncateTable,
    exp.Merge,
    exp.Grant,
    exp.Revoke,
    exp.Analyze,
)


In [14]:
sql = "SELECT * FROM air_traffic"

is_sql_allowed(
    sql,
    allowed_statements=READ_ONLY_ALLOWED,
    disallowed_statements=READ_ONLY_DISALLOWED,
    dialect="duckdb",
)


True

In [15]:
sql = "DROP TABLE air_traffic"

is_sql_allowed(
    sql,
    allowed_statements=READ_ONLY_ALLOWED,
    disallowed_statements=READ_ONLY_DISALLOWED,
    dialect="duckdb",
)


False